# Medical Data Analysis — Biochemical & Clinical Biomarkers Study

Statistical analysis of 11 biochemical and clinical variables across three study groups
(**Control**, **Newly Diagnosed Patients / NDP**, **Treated Patients / TP**), plus a
longitudinal paired comparison on a subset of 20 patients before and after treatment.

## Variables

| Group | Markers |
|-------|---------|
| Oxidative / inflammation / tumor | HO-1, CP, CEA |
| Clinical | Hb, Creatinine, ALT, Age |
| Trace elements | Cu, Fe, Zn, Ca |

## Analyses included (matching the final report)

| # | Analysis | Report reference |
|---|----------|------------------|
| 1 | Descriptive Statistics (total + per group) | Tables 1, 2 |
| 2 | Sex distribution figure | Sex figure |
| 3 | Shapiro-Wilk normality test | Methodology |
| 4 | Kruskal-Wallis H + Dunn's post-hoc | Table 3 |
| 5 | One-Way ANOVA + Tukey's post-hoc | Table 4 |
| 6 | Wilcoxon Signed-Rank Test (paired Before/After) | Treatment Efficacy table |
| 7 | Bar charts (Mean ± SD) with post-hoc letters | Figures per variable |
| 8 | Box plots (Median ± SIQR) with swarm + mean marker | Figures per variable |
| 9 | ROC curve analysis (3 pairwise comparisons) | ROC figures per variable |
| 10 | ROC results table (AUC, cut-off, sensitivity, specificity) | Report narrative |
| 11 | Spearman correlation scatter plots (overall + per group) | Correlation figures |
| 12 | Pearson & Spearman correlation heatmaps | Heatmap figures |

> **Data note:** Raw datasets are not included in this repository for confidentiality.
> The notebook expects:
> - `Data.xlsx` - sheet `Clean Data` for the main (n=95) analysis
> - `matched group1&2.xlsx` - paired Before/After data (n=20) for the Wilcoxon analysis


## 1. Setup

In [ ]:
# Non-default dependency for Dunn's / Tukey post-hoc letter assignment.
# Uncomment the next line on first run:
# !pip install scikit-posthocs

import os
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_curve, auc, confusion_matrix
import scikit_posthocs as sp

sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100


## 2. Load and Clean Data

Data is loaded **once** and reused by every downstream analysis. Cleaning steps:
- Strip whitespace and stray quotes from the `Group` column
- Rename raw group codes: `Group 1` -> Control, `Group 2` -> NDP, `Group 3` -> TP
- Standardize the `Sex` column to `Male` / `Female`
- Cast every biomarker column to numeric


In [ ]:
DATA_FILE = "Data.xlsx"
SHEET_NAME = "Clean Data"

# The 11 biomarker/clinical variables analyzed throughout the study
NUMERIC_COLS = [
    "HO-1", "CP", "CEA", "Hb", "Creatinine", "ALT",
    "Age", "Cu", "Fe", "Zn", "Ca",
]

GROUP_MAP = {
    "Group 1": "Control",
    "Group 2": "Newly Diagnosed Patients",
    "Group 3": "Treated Patients",
}
GROUP_ORDER = ["Control", "Newly Diagnosed Patients", "Treated Patients"]
GROUP_SHORT = ["Control", "NDP", "TP"]


def load_data(path: str = DATA_FILE, sheet: str = SHEET_NAME) -> pd.DataFrame:
    """Load the cleaned dataset and standardize the Group / Sex columns."""
    df = pd.read_excel(path, sheet_name=sheet)
    df.columns = df.columns.str.strip()

    df["Group"] = (
        df["Group"].astype(str).str.replace('"', "").str.replace("'", "").str.strip()
    )
    df["Group"] = df["Group"].replace(GROUP_MAP)

    if "Sex" in df.columns:
        df["Sex"] = df["Sex"].astype(str).str.strip().str.upper()
        df["Sex"] = df["Sex"].replace(
            {"M": "Male", "F": "Female", "MALE": "Male", "FEMALE": "Female"}
        )

    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


df = load_data()
print(f"Loaded {len(df)} rows | Groups: {sorted(df['Group'].unique())}")
df.head()


## 3. Shared Helpers

In [ ]:
def format_p(p: float) -> str:
    """Format a p-value with 3 decimals; suffix with * when significant."""
    if pd.isna(p):
        return "-"
    if p < 0.001:
        return "< 0.001 *"
    return f"{p:.3f}" + (" *" if p < 0.05 else "")


def shapiro_p(series: pd.Series):
    """Shapiro-Wilk p-value for the given series (returns NaN if N < 3)."""
    clean = series.dropna()
    if len(clean) < 3:
        return np.nan
    try:
        _, p = stats.shapiro(clean)
        return p
    except Exception:
        return np.nan


def get_letters(p_matrix: pd.DataFrame, groups: list) -> dict:
    """Assign compact letter display (a, b, c) from a pairwise p-value matrix.

    Groups sharing a letter are not statistically different.
    Handles the three-group case used in this study.
    """
    g1, g2, g3 = groups
    letters = {g1: "", g2: "", g3: ""}

    try:
        p12 = p_matrix.loc[g1, g2]
        p13 = p_matrix.loc[g1, g3]
        p23 = p_matrix.loc[g2, g3]
    except KeyError:
        return {g: "a" for g in groups}

    if p12 > 0.05 and p13 > 0.05 and p23 > 0.05:
        return {g: "a" for g in groups}
    if p12 <= 0.05 and p13 <= 0.05 and p23 <= 0.05:
        return {g1: "a", g2: "b", g3: "c"}

    if p12 > 0.05:
        letters[g1] += "a"; letters[g2] += "a"
    else:
        letters[g1] += "a"; letters[g2] += "b"

    match_g1 = p13 > 0.05
    match_g2 = p23 > 0.05

    if match_g1:
        letters[g3] += "a" if "a" in letters[g1] else "b"
    if match_g2:
        if "b" in letters[g2] and "b" not in letters[g3]:
            letters[g3] += "b"
        elif "a" in letters[g2] and "a" not in letters[g3]:
            letters[g3] += "a"

    if not match_g1 and not match_g2:
        letters[g3] = "c"

    for g in letters:
        letters[g] = "".join(sorted(set(letters[g]))) or "c"
    return letters


## 4. Descriptive Statistics
*Produces Tables 1 and 2 in the report.*

Mean ± SE, SD, Min, Q1, Median, Q3, Max, Range, IQR for the total sample and
stratified by study group.


In [ ]:
def descriptive_stats(series: pd.Series) -> dict:
    """Summary statistics dict for one numeric series."""
    if series.empty:
        return {}
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    return {
        "Mean ± SE": f"{series.mean():.2f} ± {series.sem():.2f}",
        "SD": round(series.std(), 2),
        "Minimum": round(series.min(), 2),
        "Q1": round(q1, 2),
        "Median": round(series.median(), 2),
        "Q3": round(q3, 2),
        "Maximum": round(series.max(), 2),
        "Range": round(series.max() - series.min(), 2),
        "IQR": round(q3 - q1, 2),
    }


# Table 1: Total sample
total_rows = []
for col in NUMERIC_COLS:
    if col not in df.columns:
        continue
    row = {"Variable": col}
    row.update(descriptive_stats(df[col].dropna()))
    total_rows.append(row)
df_total_stats = pd.DataFrame(total_rows)

# Table 2: Per group
group_rows = []
for col in NUMERIC_COLS:
    if col not in df.columns:
        continue
    for group in GROUP_ORDER:
        sub = df.loc[df["Group"] == group, col].dropna()
        row = {"Variable": col, "Group": group}
        row.update(descriptive_stats(sub))
        group_rows.append(row)
df_group_stats = pd.DataFrame(group_rows)

with pd.ExcelWriter("Descriptive_Statistics.xlsx") as writer:
    df_total_stats.to_excel(writer, sheet_name="Total", index=False)
    df_group_stats.to_excel(writer, sheet_name="By Group", index=False)

print("Saved: Descriptive_Statistics.xlsx")
df_total_stats


## 5. Sex Distribution
*Produces the sex-distribution figure in the report.*


In [ ]:
if "Sex" not in df.columns:
    raise ValueError("Dataset does not contain a 'Sex' column.")

sex_totals = df["Sex"].value_counts()

df_plot = pd.concat([df, df.assign(Group="Total")], ignore_index=True)

counts = df_plot.groupby(["Group", "Sex"]).size().reset_index(name="Count")
group_totals = counts.groupby("Group")["Count"].sum()
counts["Percentage"] = counts.apply(
    lambda r: r["Count"] / group_totals[r["Group"]], axis=1
)

group_order_with_total = GROUP_ORDER + ["Total"]

fig, ax = plt.subplots(figsize=(14, 8))
sns.barplot(
    data=counts, x="Group", y="Percentage", hue="Sex",
    order=group_order_with_total, palette="Greys",
    edgecolor="black", ax=ax,
)

ax.set_ylabel("Percentage", fontsize=14, weight="bold")
ax.set_xlabel("Study Group", fontsize=14, weight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_ylim(0, 1.15)

hue_order = sorted(counts["Sex"].unique())
for i, container in enumerate(ax.containers):
    hue_label = hue_order[i]
    sub = (counts[counts["Sex"] == hue_label]
           .set_index("Group").reindex(group_order_with_total))
    labels = [
        f"{int(r.Count)} ({r.Percentage:.1%})" if pd.notna(r.Count) else "0"
        for r in sub.itertuples()
    ]
    ax.bar_label(container, labels=labels, fontsize=10, weight="bold", padding=3)

legend_handles = []
for i, sex in enumerate(hue_order):
    total = sex_totals.get(sex, 0)
    color = "#D3D3D3" if i == 0 else "#808080"
    legend_handles.append(mpatches.Patch(
        facecolor=color, edgecolor="black", label=f"{sex} (Total: {total})"
    ))
ax.legend(handles=legend_handles, title="Sex", loc="upper right",
          fontsize=11, title_fontsize=12)

plt.xticks(fontsize=11)
plt.tight_layout()
plt.savefig("sex_distribution.png", dpi=300)
plt.show()


## 6. Shapiro-Wilk Normality Test

Per-variable, per-group assessment of normality. The overall-sample p-value
is what appears in the **Normality** column of Tables 3 and 4.


In [ ]:
normality_rows = []

for col in NUMERIC_COLS:
    if col not in df.columns:
        continue
    row = {"Variable": col}

    all_groups_normal = True
    for g in GROUP_ORDER:
        p = shapiro_p(df.loc[df["Group"] == g, col])
        row[f"{g} p-value"] = format_p(p) if not pd.isna(p) else "N<3"
        if pd.isna(p) or p <= 0.05:
            all_groups_normal = False

    p_total = shapiro_p(df[col])
    row["Total p-value"] = format_p(p_total) if not pd.isna(p_total) else "N<3"

    row["Recommendation"] = (
        "Parametric (ANOVA)" if all_groups_normal
        else "Non-Parametric (Kruskal-Wallis)"
    )
    normality_rows.append(row)

normality_df = pd.DataFrame(normality_rows)
normality_df.to_excel("Shapiro_Wilk_Normality.xlsx", index=False)

print("Saved: Shapiro_Wilk_Normality.xlsx")
normality_df


## 7. Kruskal-Wallis H Test with Dunn's Post-Hoc
*Produces Table 3 in the report.*

Post-hoc pairwise contrasts use Dunn's test with Bonferroni correction.
The **Normality** column shows Shapiro-Wilk on the total sample.


In [ ]:
kruskal_results = []

for col in NUMERIC_COLS:
    if col not in df.columns:
        continue

    group_data = [df.loc[df["Group"] == g, col].dropna() for g in GROUP_ORDER]
    if any(len(d) < 2 for d in group_data):
        continue

    norm_p = shapiro_p(df[col])
    norm_res = format_p(norm_p) if not pd.isna(norm_p) else "-"

    h_stat, p_k = stats.kruskal(*group_data)

    letter_map = {g: "a" for g in GROUP_ORDER}
    if p_k < 0.05:
        try:
            p_matrix = sp.posthoc_dunn(
                df[["Group", col]].dropna(),
                val_col=col, group_col="Group", p_adjust="bonferroni",
            )
            letter_map = get_letters(p_matrix, GROUP_ORDER)
        except Exception:
            pass

    row = {"Variable": col}
    for g, data in zip(GROUP_ORDER, group_data):
        q1, q3 = data.quantile(0.25), data.quantile(0.75)
        siqr = (q3 - q1) / 2
        row[f"Median ± SIQR ({g})"] = (
            f"{data.median():.2f} ± {siqr:.2f} {letter_map[g]}"
        )

    row["Normality (Shapiro-Wilk)"] = norm_res
    row["H_Statistic"] = round(h_stat, 3)
    row["Kruskal_P_Value"] = format_p(p_k)
    kruskal_results.append(row)

kruskal_df = pd.DataFrame(kruskal_results)
kruskal_df.to_excel("Kruskal_Analysis.xlsx", index=False)

print(f"Saved Kruskal_Analysis.xlsx - {len(kruskal_df)} variables analyzed")
kruskal_df


## 8. One-Way ANOVA with Tukey's Post-Hoc
*Produces Table 4 in the report.*


In [ ]:
anova_results = []

for col in NUMERIC_COLS:
    if col not in df.columns:
        continue

    group_data = [df.loc[df["Group"] == g, col].dropna() for g in GROUP_ORDER]
    if any(len(d) < 2 for d in group_data):
        continue

    norm_p = shapiro_p(df[col])
    norm_res = format_p(norm_p) if not pd.isna(norm_p) else "-"

    f_stat, p_anova = stats.f_oneway(*group_data)

    letter_map = {g: "a" for g in GROUP_ORDER}
    if p_anova < 0.05:
        try:
            p_matrix = sp.posthoc_tukey(
                df[["Group", col]].dropna(), val_col=col, group_col="Group"
            )
            letter_map = get_letters(p_matrix, GROUP_ORDER)
        except Exception:
            pass

    row = {"Variable": col}
    for g, data in zip(GROUP_ORDER, group_data):
        row[f"Mean ± SD ({g})"] = (
            f"{data.mean():.2f} ± {data.std():.2f} {letter_map[g]}"
        )

    row["Normality (Shapiro-Wilk)"] = norm_res
    row["F-ratio"] = f"{f_stat:.2f}"
    row["P-value"] = format_p(p_anova)
    anova_results.append(row)

anova_df = pd.DataFrame(anova_results)
anova_df.to_excel("ANOVA_Results.xlsx", index=False)

print(f"Saved ANOVA_Results.xlsx - {len(anova_df)} variables analyzed")
anova_df


## 9. Wilcoxon Signed-Rank Test - Treatment Efficacy
*Produces the Treatment Efficacy table in the report.*

Paired non-parametric comparison on the longitudinal subset (**n = 20**).

Expected input file: `matched group1&2.xlsx` - rows for "Newly Diagnosed" are stacked
above rows for "Treated", separated by a row containing the word "Group".


In [ ]:
PAIRED_FILE = "matched group1&2.xlsx"


def get_median_iqr(series: pd.Series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    return series.median(), q3 - q1


def wilcoxon_significance(p: float) -> str:
    if pd.isna(p):
        return "Error"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "NS"


def run_wilcoxon(path: str = PAIRED_FILE) -> pd.DataFrame:
    """Parse stacked Before/After file and run paired Wilcoxon per variable."""
    df_raw = (pd.read_csv(path, header=None) if path.endswith(".csv")
              else pd.read_excel(path, header=None))

    group_rows = df_raw[
        df_raw[0].astype(str).str.contains("Group", case=False, na=False)
    ].index.tolist()
    if len(group_rows) < 2:
        raise ValueError("Expected at least 2 'Group' marker rows in the file.")

    idx_hdr_b = group_rows[0] + 1
    idx_end_b = group_rows[1]
    df_before = df_raw.iloc[idx_hdr_b + 1 : idx_end_b].copy()
    df_before.columns = df_raw.iloc[idx_hdr_b]

    idx_hdr_a = group_rows[1] + 1
    df_after = df_raw.iloc[idx_hdr_a + 1:].copy()
    df_after.columns = df_raw.iloc[idx_hdr_a]

    df_before = df_before.dropna(how="all")
    df_after = df_after.dropna(how="all")

    for sub in (df_before, df_after):
        if "Name" in sub.columns and sub["Name"].isnull().mean() > 0.5:
            sub["Name"] = sub[sub.columns[1]]

    for sub in (df_before, df_after):
        sub["Name"] = sub["Name"].astype(str).str.strip()
        sub.columns = sub.columns.str.strip()

    merged = pd.merge(df_before, df_after, on="Name", suffixes=("_Before", "_After"))

    exclude = {"Name", "Sex", "Group", "id", "ID", "nan"}
    candidates = [
        c.replace("_Before", "") for c in merged.columns if "_Before" in c
    ]
    target_vars = [c for c in candidates if c not in exclude]

    results = []
    for var in target_vars:
        c_pre, c_post = f"{var}_Before", f"{var}_After"
        merged[c_pre] = pd.to_numeric(merged[c_pre], errors="coerce")
        merged[c_post] = pd.to_numeric(merged[c_post], errors="coerce")
        valid = merged[[c_pre, c_post]].dropna()
        if len(valid) < 2:
            continue

        med_b, iqr_b = get_median_iqr(valid[c_pre])
        med_a, iqr_a = get_median_iqr(valid[c_post])

        diff = valid[c_pre] - valid[c_post]
        if (diff == 0).all():
            stat_val, p_val, sig = 0.0, 1.0, "Identical"
        else:
            try:
                stat_val, p_val = stats.wilcoxon(valid[c_pre], valid[c_post])
                sig = wilcoxon_significance(p_val)
            except Exception:
                stat_val, p_val, sig = np.nan, np.nan, "Error"

        results.append({
            "Variable": var,
            "Newly Diagnosed (Median ± IQR)": f"{med_b:.2f} ± {iqr_b:.2f}",
            "Treated (Median ± IQR)": f"{med_a:.2f} ± {iqr_a:.2f}",
            "Wilcoxon Stat": f"{stat_val:.1f}",
            "P-value": "< 0.001" if (not pd.isna(p_val) and p_val < 0.001)
                       else f"{p_val:.5f}",
            "Sig.": sig,
        })

    return pd.DataFrame(results)


if os.path.exists(PAIRED_FILE):
    wilcoxon_df = run_wilcoxon()
    wilcoxon_df.to_excel("Wilcoxon_Test_Report.xlsx", index=False)
    print(f"Saved Wilcoxon_Test_Report.xlsx - {len(wilcoxon_df)} variables")
    wilcoxon_df
else:
    print(f"File '{PAIRED_FILE}' not found - skipping paired analysis.")


## 10. Bar Charts with Post-Hoc Letters
*Produces the Mean ± SD figures in the report.*


In [ ]:
BAR_COLORS = ["#D3D3D3", "#A9A9A9", "#808080"]
BAR_PATTERNS = ["/", "\\", "x"]

legend_handles = [
    mpatches.Patch(facecolor=BAR_COLORS[i], hatch=BAR_PATTERNS[i],
                   edgecolor="black",
                   label=f"{GROUP_SHORT[i]}: {GROUP_ORDER[i]}")
    for i in range(3)
]

os.makedirs("figures/bar_charts", exist_ok=True)
mean_sd_cols = [f"Mean ± SD ({g})" for g in GROUP_ORDER]

for _, row in anova_df.iterrows():
    var_name = row["Variable"]

    means, sds, letters = [], [], []
    for col in mean_sd_cols:
        val_str = str(row[col])
        # Format: "X.XX ± Y.YY letter"
        try:
            parts = val_str.split()
            means.append(float(parts[0]))
            sds.append(float(parts[2]))
            letters.append(parts[3] if len(parts) > 3 else "")
        except (ValueError, IndexError):
            means.append(0); sds.append(0); letters.append("")

    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.bar(
        GROUP_SHORT, means, yerr=sds, capsize=5,
        color=BAR_COLORS, edgecolor="black", linewidth=1.0,
    )
    for bar, pat in zip(bars, BAR_PATTERNS):
        bar.set_hatch(pat)

    ax.set_xlabel("Group", fontsize=12, fontweight="bold")
    ax.set_ylabel(f"Mean {var_name}", fontsize=12, fontweight="bold")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)

    max_y = max((m + s) for m, s in zip(means, sds)) or 1
    ax.set_ylim(0, max_y * 1.35)

    for i, bar in enumerate(bars):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + sds[i] + max_y * 0.02,
            f"({letters[i]})\nMean: {means[i]:.2f}\nSD: {sds[i]:.2f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold",
        )

    ax.legend(handles=legend_handles, loc="upper right", fontsize=10,
              frameon=True, edgecolor="black", framealpha=1, fancybox=False)

    plt.tight_layout()
    safe_name = "".join(c for c in var_name if c.isalnum() or c in " -_").strip()
    plt.savefig(f"figures/bar_charts/BarChart_{safe_name}.png", dpi=300)
    plt.close()

print(f"Saved {len(anova_df)} bar charts to figures/bar_charts/")


## 11. Box Plots with Swarm Overlay and Mean Marker
*Produces the Median ± SIQR figures in the report.*

Each plot shows: the IQR box, median line, whiskers, all individual data points
(jittered swarm), and the mean (red diamond). Post-hoc letters from Dunn's test
are placed above each whisker cap.


In [ ]:
os.makedirs("figures/box_plots", exist_ok=True)
median_cols = [f"Median ± SIQR ({g})" for g in GROUP_ORDER]

for _, row in kruskal_df.iterrows():
    var_name = row["Variable"]
    if var_name not in df.columns:
        continue

    data_per_group = [
        df.loc[df["Group"] == g, var_name].dropna().values for g in GROUP_ORDER
    ]

    # Extract trailing letter from each "Median ± SIQR (Group)" cell
    letters = []
    for col in median_cols:
        cell_str = str(row[col]).strip()
        parts = cell_str.split()
        # Last token is the letter if it's non-numeric
        last = parts[-1] if parts else ""
        try:
            float(last)
            letters.append("")
        except ValueError:
            letters.append(last)

    fig, ax = plt.subplots(figsize=(10, 7))

    bplot = ax.boxplot(
        data_per_group, patch_artist=True, labels=GROUP_SHORT,
        widths=0.6, showmeans=False,
        flierprops={"marker": "o", "markerfacecolor": "black", "markersize": 4},
    )
    for patch, color, pat in zip(bplot["boxes"], BAR_COLORS, BAR_PATTERNS):
        patch.set_facecolor(color)
        patch.set_hatch(pat)
        patch.set_edgecolor("black")
    for element in ["whiskers", "caps", "medians"]:
        plt.setp(bplot[element], color="black", linewidth=1.2)

    # Manual jittered scatter aligned with boxplot positions (1, 2, 3)
    rng = np.random.default_rng(0)
    for i, data in enumerate(data_per_group):
        jitter = rng.uniform(-0.12, 0.12, size=len(data))
        ax.scatter(
            np.full(len(data), i + 1) + jitter, data,
            color="black", s=14, alpha=0.55, zorder=5,
        )

    # Mean markers (red diamonds)
    means = [np.mean(d) if len(d) else np.nan for d in data_per_group]
    ax.scatter(range(1, len(GROUP_ORDER) + 1), means,
               marker="D", color="red", s=80, zorder=10,
               edgecolor="black", linewidth=0.8)

    ax.set_xlabel("Group", fontsize=12, fontweight="bold")
    ax.set_ylabel(var_name, fontsize=12, fontweight="bold")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)

    y_min, y_max = ax.get_ylim()
    offset = (y_max - y_min) * 0.03
    for i, letter in enumerate(letters):
        if not letter:
            continue
        top_whisker = bplot["caps"][(i * 2) + 1].get_ydata()[0]
        ax.text(i + 1, top_whisker + offset, f"({letter})",
                ha="center", va="bottom", fontsize=12, fontweight="bold")

    legend_handles_box = legend_handles + [
        plt.Line2D([0], [0], marker="D", color="w", markerfacecolor="red",
                   markeredgecolor="black", markersize=10, label="Mean"),
    ]
    ax.legend(handles=legend_handles_box, loc="upper right", fontsize=9,
              frameon=True, edgecolor="black", framealpha=1, fancybox=False)

    plt.tight_layout()
    safe_name = "".join(c for c in var_name if c.isalnum() or c in " -_").strip()
    plt.savefig(f"figures/box_plots/BoxPlot_{safe_name}.png", dpi=300)
    plt.close()

print(f"Saved {len(kruskal_df)} box plots to figures/box_plots/")


## 12. ROC Curve Analysis - Plots
*Produces one ROC figure per variable, with three pairwise comparisons each.*

For every biomarker we evaluate its discrimination in three contexts:
Control vs NDP, Control vs TP, and TP vs NDP. The black cross marks the
Youden J-optimal cutoff on each curve.


In [ ]:
# (neg_group, pos_group, short_label)
ROC_COMPARISONS = [
    ("Control", "Newly Diagnosed Patients", "Control vs NDP"),
    ("Control", "Treated Patients",          "Control vs TP"),
    ("Treated Patients", "Newly Diagnosed Patients", "TP vs NDP"),
]
ROC_COLORS = ["#1f77b4", "#2ca02c", "#d62728"]
ROC_LINESTYLES = ["-", "--", "-."]
ROC_MARKERS = ["o", "^", "s"]

os.makedirs("figures/roc_curves", exist_ok=True)

for var in NUMERIC_COLS:
    if var not in df.columns:
        continue

    fig, ax = plt.subplots(figsize=(10, 8))
    plotted = False

    for i, (neg_g, pos_g, label) in enumerate(ROC_COMPARISONS):
        sub = df[df["Group"].isin([neg_g, pos_g])].dropna(subset=[var])
        if len(sub) < 4 or sub[var].nunique() < 2:
            continue
        plotted = True

        y_true = (sub["Group"] == pos_g).astype(int)
        y_scores = sub[var].values

        fpr, tpr, thresholds = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)

        # Flip direction if AUC < 0.5 (marker is inversely predictive)
        if roc_auc < 0.5:
            y_scores = -y_scores
            fpr, tpr, thresholds = roc_curve(y_true, y_scores)
            roc_auc = auc(fpr, tpr)

        # Youden J optimal cutoff
        ix = np.argmax(tpr - fpr)
        best_thresh = thresholds[ix]
        y_pred = (y_scores >= best_thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0

        mid = len(fpr) // 2
        ax.plot(
            fpr, tpr,
            color=ROC_COLORS[i], linestyle=ROC_LINESTYLES[i], marker=ROC_MARKERS[i],
            markevery=[mid], markersize=10, lw=2.5,
            label=f"{label} (AUC={roc_auc:.2f})\nSens:{sens:.1%} Spec:{spec:.1%}",
        )
        ax.scatter(fpr[ix], tpr[ix], marker="X", color="black", s=100, zorder=10)

    if not plotted:
        plt.close(fig)
        continue

    ax.plot([0, 1], [0, 1], color="gray", lw=1.5, linestyle=":")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("False Positive Rate (1 - Specificity)",
                  fontsize=12, fontweight="bold")
    ax.set_ylabel("True Positive Rate (Sensitivity)",
                  fontsize=12, fontweight="bold")
    ax.legend(loc="lower right", fontsize=10, frameon=True, framealpha=0.9)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    safe = str(var).replace("/", "-").replace("\\", "-")
    plt.savefig(f"figures/roc_curves/ROC_{safe}.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Saved ROC curves to figures/roc_curves/")


## 13. ROC Results Table
*Produces the AUC / cutoff / sensitivity / specificity values cited in the ROC narrative.*


In [ ]:
roc_rows = []

for var in NUMERIC_COLS:
    if var not in df.columns:
        continue

    for neg_g, pos_g, label in ROC_COMPARISONS:
        sub = df[df["Group"].isin([neg_g, pos_g])].dropna(subset=[var])
        if len(sub) < 4 or sub[var].nunique() < 2:
            continue

        y_true = (sub["Group"] == pos_g).astype(int)
        y_scores = sub[var].values

        fpr, tpr, thresholds = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)

        is_inverted = False
        if roc_auc < 0.5:
            y_scores = -y_scores
            fpr, tpr, thresholds = roc_curve(y_true, y_scores)
            roc_auc = auc(fpr, tpr)
            is_inverted = True

        ix = np.argmax(tpr - fpr)
        best_thresh = thresholds[ix]
        display_thresh = -best_thresh if is_inverted else best_thresh

        y_pred = (y_scores >= best_thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0
        acc = (tp + tn) / (tp + tn + fp + fn)

        roc_rows.append({
            "Variable": var,
            "Comparison": label,
            "AUC": round(roc_auc, 3),
            "Cut-off Value": round(display_thresh, 3),
            "Sensitivity (%)": round(sens * 100, 1),
            "Specificity (%)": round(spec * 100, 1),
            "Accuracy (%)": round(acc * 100, 1),
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        })

roc_results_df = pd.DataFrame(roc_rows)
roc_results_df.to_excel("ROC_Analysis_Results.xlsx", index=False)

print(f"Saved ROC_Analysis_Results.xlsx - {len(roc_results_df)} rows")
roc_results_df.head(12)


## 14. Spearman Correlation - Scatter Plots (Overall + per Group)
*Produces the per-pair correlation figures in the report.*

Each figure is a 2x2 grid: total sample + one subplot per study group.
Report-highlighted pairs: **HO-1 vs CP, HO-1 vs CEA, CP vs CEA, HO-1 vs Zn, CP vs Zn**.


In [ ]:
def plot_spearman_panel(ax, data: pd.DataFrame, x_var: str, y_var: str,
                        title: str, color: str):
    valid = data[[x_var, y_var]].apply(pd.to_numeric, errors="coerce").dropna()
    if len(valid) < 2:
        ax.text(0.5, 0.5, "Not enough data",
                transform=ax.transAxes, ha="center", va="center")
        ax.set_title(title, fontsize=11, weight="bold")
        return

    r, p = spearmanr(valid[x_var], valid[y_var])
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

    sns.regplot(
        data=valid, x=x_var, y=y_var, ax=ax,
        scatter_kws={"alpha": 0.6, "color": color, "s": 50},
        line_kws={"color": "black", "linestyle": "--"}, ci=95,
    )
    ax.text(
        0.05, 0.95, f"Spearman rho = {r:.3f}\n{p_text}\nn = {len(valid)}",
        transform=ax.transAxes, va="top", ha="left", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.85, ec="gray"),
    )
    ax.set_title(title, fontsize=11, weight="bold")
    ax.set_xlabel(x_var, fontsize=11)
    ax.set_ylabel(y_var, fontsize=11)


SCATTER_PAIRS = [
    ("HO-1", "CP"),
    ("HO-1", "CEA"),
    ("CP",   "CEA"),
    ("HO-1", "Zn"),
    ("CP",   "Zn"),
]
GROUP_COLORS = ["forestgreen", "goldenrod", "firebrick"]

os.makedirs("figures/correlations", exist_ok=True)

for x_var, y_var in SCATTER_PAIRS:
    if x_var not in df.columns or y_var not in df.columns:
        print(f"Skipping {x_var} vs {y_var} - column missing")
        continue

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    plot_spearman_panel(axes[0, 0], df, x_var, y_var,
                        f"Total Sample (n = {df[[x_var, y_var]].dropna().shape[0]})",
                        "steelblue")
    for i, group in enumerate(GROUP_ORDER):
        ax = axes[(i + 1) // 2, (i + 1) % 2]
        sub = df[df["Group"] == group]
        plot_spearman_panel(ax, sub, x_var, y_var, group, GROUP_COLORS[i])

    fig.suptitle(f"Spearman Correlation: {x_var} vs {y_var}",
                 fontsize=14, weight="bold")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    safe = f"{x_var}_{y_var}".replace(" ", "_").replace("/", "_")
    plt.savefig(f"figures/correlations/spearman_{safe}.png", dpi=300)
    plt.close()

print(f"Saved {len(SCATTER_PAIRS)} correlation scatter figures")


## 15. Correlation Heatmaps - Pearson & Spearman
*Produces the heatmap figures in the report.*


In [ ]:
def compute_corr_matrix(data: pd.DataFrame, method: str = "pearson"):
    """Return (correlation matrix, annotation matrix, excel-friendly matrix)."""
    cols = list(data.columns)
    n = len(cols)
    corr = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)
    annot = pd.DataFrame(np.empty((n, n), dtype=object), index=cols, columns=cols)
    excel = pd.DataFrame(np.empty((n, n), dtype=object), index=cols, columns=cols)

    fn = pearsonr if method == "pearson" else spearmanr

    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            if i == j:
                corr.iloc[i, j] = 1.0
                annot.iloc[i, j] = "1.00"
                excel.iloc[i, j] = "1.00"
                continue
            pair = data[[c1, c2]].dropna()
            if len(pair) < 2:
                corr.iloc[i, j] = 0
                annot.iloc[i, j] = "NaN"
                excel.iloc[i, j] = "NaN"
                continue
            r, p = fn(pair[c1], pair[c2])
            corr.iloc[i, j] = r
            p_text = "p<0.001" if p < 0.001 else f"p={p:.3f}"
            annot.iloc[i, j] = f"{r:.2f}\n({p_text})"
            excel.iloc[i, j] = f"{r:.3f} ({p_text})"

    return corr, annot, excel


df_numeric = df[[c for c in NUMERIC_COLS if c in df.columns]]

r_pearson, annot_pearson, excel_pearson = compute_corr_matrix(df_numeric, "pearson")
r_spearman, annot_spearman, excel_spearman = compute_corr_matrix(df_numeric, "spearman")

for name, corr_mat, annot_mat in [
    ("pearson", r_pearson, annot_pearson),
    ("spearman", r_spearman, annot_spearman),
]:
    plt.figure(figsize=(14, 12))
    sns.heatmap(
        corr_mat, annot=annot_mat, fmt="",
        cmap="coolwarm", center=0, linewidths=0.5,
        cbar_kws={"shrink": 0.8}, annot_kws={"size": 9},
    )
    plt.tight_layout()
    plt.savefig(f"heatmap_{name}.png", dpi=300)
    plt.show()

with pd.ExcelWriter("correlation_analysis.xlsx") as writer:
    excel_pearson.to_excel(writer, sheet_name="Pearson")
    excel_spearman.to_excel(writer, sheet_name="Spearman")

print("Saved: heatmap_pearson.png, heatmap_spearman.png, correlation_analysis.xlsx")
